# 3D Gaussian Splatting — Google Colab 학습

이 노트북은 [hojunkim02/gaussian-orgin](https://github.com/hojunkim02/gaussian-orgin) 레포지토리(3D Gaussian Splatting)를 Google Colab GPU 환경에서 학습하기 위한 것입니다.

**사용 방법**
1. `런타임 > 런타임 유형 변경`에서 **GPU(T4 권장)**를 선택하세요.
2. 위에서부터 순서대로 셀을 실행하세요.
3. 학습 결과는 `output/`에 저장되며, 마지막 셀에서 `.zip`으로 다운로드할 수 있습니다.

`DATA_PATH`를 자신의 데이터셋 경로로 바꾸면 자신의 데이터로 학습할 수 있습니다. 전체 학습(30K 반복)은 T4 기준 수 시간 소요되므로, 테스트 시에는 `ITERATIONS`를 줄이세요.

In [ ]:
# GPU 확인
!nvidia-smi
import torch
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요: 런타임 > 런타임 유형 변경 > GPU'

In [ ]:
# 레포지토리 클론 및 서브모듈 초기화
REPO_URL = "https://github.com/hojunkim02/gaussian-orgin"
%cd /content
![ -d gaussian-orgin ] || git clone --recursive $REPO_URL
%cd gaussian-orgin
!git submodule sync
!git submodule update --init --recursive --force

In [ ]:
# CUDA 아키텍처 설정 (Colab GPU에 맞게 자동 감지)
import torch, os
gpu_name = torch.cuda.get_device_name(0).upper()
if "T4" in gpu_name:
    cuda_arch = "75"
elif "A100" in gpu_name:
    cuda_arch = "80"
elif "L4" in gpu_name:
    cuda_arch = "89"
elif "V100" in gpu_name:
    cuda_arch = "70"
else:
    cuda_arch = "75"
print(f"GPU: {gpu_name}, TORCH_CUDA_ARCH_LIST={cuda_arch}")
os.environ["TORCH_CUDA_ARCH_LIST"] = cuda_arch

In [ ]:
# 서브모듈 빌드 및 설치 (몇 분 소요)
!pip install -q plyfile tqdm opencv-python joblib
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn
!pip install -q submodules/fused-ssim

In [ ]:
# 데이터셋 준비
# 기본값: 공식 예제 데이터셋(tandt_db, 약 650MB) 중 truck 장면 사용
# 자신의 데이터를 쓰려면 USE_EXAMPLE_DATA=False 로 두고 DATA_PATH를 지정하세요.
DATA_PATH = ""  # @param {type:"string"}
USE_EXAMPLE_DATA = True  # @param {type:"boolean"}

if USE_EXAMPLE_DATA:
    %cd /content/gaussian-orgin
    import os
    if not os.path.isdir("data/tandt"):
        os.makedirs("data", exist_ok=True)
        !wget -q https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O data/tandt_db.zip
        !unzip -q data/tandt_db.zip -d data && rm data/tandt_db.zip
    DATA_PATH = "/content/gaussian-orgin/data/tandt/truck"

print("데이터 경로:", DATA_PATH)
assert os.path.isdir(DATA_PATH), f"데이터 경로가 존재하지 않습니다: {DATA_PATH}"

In [ ]:
# (선택) 구글 드라이브 마운트 — 자신의 데이터셋 로드 또는 결과 저장용
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 학습 실행
# ITERATIONS: Colab 세션 길이에 맞게 조절 (예: 테스트 7000 / 전체 30000)
ITERATIONS = 30000  # @param {type:"integer"}
SAVE_ITER = 7000    # @param {type:"integer"}
OUTPUT_DIR = "/content/gaussian-orgin/output"

%cd /content/gaussian-orgin
!python train.py -s $DATA_PATH -m $OUTPUT_DIR --iterations $ITERATIONS --save_iterations $SAVE_ITER --test_iterations $SAVE_ITER

In [ ]:
# 렌더링 (테스트/렌더 카메라 뷰)
%cd /content/gaussian-orgin
!python render.py -m $OUTPUT_DIR

In [ ]:
# (선택) 지표 계산 (PSNR / SSIM / LPIPS)
%cd /content/gaussian-orgin
!pip install -q lpips
!python metrics.py -m $OUTPUT_DIR

In [ ]:
# 렌더링 결과를 GIF로 미리보기
import glob, imageio, os
renders = sorted(glob.glob('/content/gaussian-orgin/output/train/renders/*.png'))
if renders:
    frames = [imageio.imread(p) for p in renders[:60]]
    imageio.mimsave('/content/preview.gif', frames, duration=0.1)
    from IPython.display import Image, display
    display(Image('/content/preview.gif'))
else:
    print("렌더링 결과가 없습니다. render.py 셀을 먼저 실행하세요.")

In [ ]:
# 학습 결과(모델 .ply 포함) 다운로드
import shutil
from google.colab import files
shutil.make_archive('/content/gaussian_output', 'zip', '/content/gaussian-orgin/output')
files.download('/content/gaussian_output.zip')